In [ ]:
# 수치 계산에 사용할 라이브러리
import numpy as np
# 표 형태 데이터를 다루기 위한 라이브러리
import pandas as pd
# 그래프를 그리기 위한 라이브러리
import matplotlib.pyplot as plt
# 변화점 탐지를 위한 라이브러리
import ruptures as rpt


In [ ]:
# 같은 결과가 나오도록 난수 시드 고정
np.random.seed(1)
# 평균이 다른 두 구간을 이어 붙여 예제 시계열 생성
m1 = np.concatenate([
    np.random.normal(0, 1, 100),
    np.random.normal(5, 1, 100)
])
# 시계열의 순서를 나타내는 인덱스 생성
time = np.arange(len(m1))

# 변화점을 찾을 알고리즘 학습
algo = rpt.Pelt(model="rbf").fit(m1)
# penalty 값을 기준으로 변화점 예측
changepoints = algo.predict(pen=10)

# 찾은 변화점 위치 출력
print(f"변화점: {changepoints}")
# 마지막 값은 종료점이라 1을 빼고 변화 횟수 계산
print(f"변화 횟수: {len(changepoints) - 1}개")


In [ ]:
# 데이터 처리를 위한 라이브러리
import pandas as pd
# 변화점 탐지 라이브러리
import ruptures as rpt
# pkl 파일을 읽을 때 사용하는 라이브러리
import pickle

# 원래 자료에 나온 pkl 로드 방식
# with open('har_total.pkl', 'rb') as f:
#     har_data = pickle.load(f)
# 현재 폴더에 있는 csv 압축 파일로 데이터 로드
har_data = pd.read_csv('A_DeviceMotion_data 복사본/HAR_total.compat.csv.gz', compression='gzip', low_memory=False)

# 그룹화할 때 문제 없도록 문자열로 변환
har_data['id'] = har_data['id'].astype(str)
har_data['exp_no'] = har_data['exp_no'].astype(str)


In [ ]:
# 사람 id, 실험 번호, 활동별로 데이터 묶기
experiments = har_data.groupby(['id', 'exp_no', 'activity'])
# 변화점 결과를 저장할 빈 리스트
ch_pt_results = []

# 각 실험 그룹마다 변화점 특징 계산
for (subject_id, exp_no, activity), group_data in experiments:
    # 회전 크기 신호 추출
    mag_rotation = group_data['magrotationRate'].values
    # 가속도 크기 신호 추출
    mag_acceleration = group_data['maguserAcceleration'].values

    # 평균 변화 개수 계산
    algo_mean = rpt.Binseg(model="l2", min_size=10, jump=5)
    cp1 = len(algo_mean.fit(mag_rotation).predict(pen=20)) - 1
    cp2 = len(algo_mean.fit(mag_acceleration).predict(pen=20)) - 1

    # 분산 변화 개수 계산
    algo_var = rpt.Binseg(model="normal", min_size=10, jump=5)
    cp3 = len(algo_var.fit(mag_rotation).predict(pen=20)) - 1
    cp4 = len(algo_var.fit(mag_acceleration).predict(pen=20)) - 1

    # 평균과 분산이 함께 변하는 개수 계산
    algo_meanvar = rpt.Binseg(model="rbf", min_size=10, jump=5)
    cp5 = len(algo_meanvar.fit(mag_rotation).predict(pen=20)) - 1
    cp6 = len(algo_meanvar.fit(mag_acceleration).predict(pen=20)) - 1

    # 한 실험의 결과를 딕셔너리로 저장
    ch_pt_results.append({
        'id': subject_id,
        'exp_no': exp_no,
        'activity': activity,
        'cp1': cp1,
        'cp2': cp2,
        'cp3': cp3,
        'cp4': cp4,
        'cp5': cp5,
        'cp6': cp6,
    })

# 리스트를 표 형태로 변환
changepoint_features = pd.DataFrame(ch_pt_results)

# 나중에 다시 쓰기 위해 csv 파일로 저장
changepoint_features.to_csv('changepoint_features.csv', index=False)
# 상위 5개 행 확인
changepoint_features.head()


In [ ]:
# 데이터 처리용 라이브러리
import pandas as pd
# 수치 계산용 라이브러리
import numpy as np
# 같은 사람 데이터가 섞이지 않게 나누는 함수
from sklearn.model_selection import GroupShuffleSplit
# 변수 크기를 맞추는 함수
from sklearn.preprocessing import StandardScaler

# 변화점 특징 파일 불러오기
df = pd.read_csv('changepoint_features.csv')

# 입력 변수 6개 선택
X = df[['cp1', 'cp2', 'cp3', 'cp4', 'cp5', 'cp6']].values
# 맞춰야 할 정답 라벨
y = df['activity'].values
# 같은 사람 데이터를 같은 그룹으로 묶기 위한 id
groups = df['id'].astype(str).values

# 같은 사람의 데이터가 train/test에 동시에 들어가지 않도록 분할
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
# 학습용 데이터와 테스트용 데이터 생성
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# 변수 크기를 비슷하게 맞춤
scaler = StandardScaler()
# 학습 데이터 기준으로 표준화 학습 후 변환
X_train_scaled = scaler.fit_transform(X_train)
# 같은 기준으로 테스트 데이터 변환
X_test_scaled = scaler.transform(X_test)


In [ ]:
# 랜덤포레스트 모델
from sklearn.ensemble import RandomForestClassifier
# XGBoost 모델
from xgboost import XGBClassifier
# LightGBM 모델
from lightgbm import LGBMClassifier
# 교차검증 함수
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 비교할 모델 3개 정의
models = {
    'RF': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'XGB': XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        random_state=42
    ),
    'LGBM': LGBMClassifier(
        n_estimators=400,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}

# 데이터를 5등분해서 성능을 평균내는 설정
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 모델별 평균 정확도를 저장할 딕셔너리
cv_scores = {}

# 각 모델의 교차검증 정확도 계산
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_scores[name] = scores.mean()

# 평균 정확도가 가장 높은 모델 선택
best_model_name = max(cv_scores, key=cv_scores.get)
# 선택된 모델 객체 저장
best_model = models[best_model_name]

# 최고 성능 모델 이름과 정확도 출력
print(f"최고 모델: {best_model_name} (정확도: {cv_scores[best_model_name]:.3f})")


In [ ]:
# 분류 성능 평가에 필요한 함수들
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# 선택된 최고 모델 학습
best_model.fit(X_train, y_train)
# 테스트 데이터 예측
y_pred = best_model.predict(X_test)

# 전체 정확도 출력
print('accuracy :', round(accuracy_score(y_test, y_pred), 4))
# 클래스별 균형을 반영한 macro f1 출력
print('macro_f1 :', round(f1_score(y_test, y_pred, average='macro'), 4))
# 클래스별 precision, recall, f1-score 출력
print(classification_report(y_test, y_pred))

# 어떤 클래스를 헷갈렸는지 혼동행렬로 확인
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, xticks_rotation=45, cmap='Blues')
plt.show()


In [ ]:
# 최종 모델에서 변수 중요도 가져오기
importance = pd.Series(best_model.feature_importances_, index=['cp1', 'cp2', 'cp3', 'cp4', 'cp5', 'cp6'])

# 중요도가 큰 순서대로 정렬
importance = importance.sort_values(ascending=False)
# 숫자로 먼저 확인
print(importance)

# 그래프로 중요도 비교
plt.figure(figsize=(8, 4))
plt.barh(importance.index, importance.values)
plt.gca().invert_yaxis()
plt.title(f'Feature Importance - {best_model_name}')
plt.xlabel('importance')
plt.ylabel('feature')
plt.show()


In [ ]:
# 피크 분석에서 만든 특징 불러오기
peak_features = pd.read_csv('peak_features.csv')
# 변화점 특징과 공통 열 기준으로 합치기
combined_features = peak_features.merge(changepoint_features, on=['id', 'exp_no', 'activity'], how='inner')
# 합쳐진 결과를 새 csv로 저장
combined_features.to_csv('peak_changepoint_features.csv', index=False)
# 결과 확인
combined_features.head()
